# 03 · Explanations, faithfulness and the negative control

Three saliency methods, then the two measurements that decide whether any of them should be
believed: a faithfulness test against a random-ranking control, and the cascading-randomisation
sanity check.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))  # works without installing the package

CONFIG = "../configs/default.yaml"   # switch to ../configs/smoke.yaml to run offline in seconds


In [ ]:
from ctxaiqc.data import load_dataset
from ctxaiqc.evaluate import load_checkpoint, predict_probs
from ctxaiqc.explain import EXPLAINERS, explain
from ctxaiqc.utils import load_config

cfg = load_config(CONFIG)
data = load_dataset(**cfg["dataset"], seed=cfg["seed"])
model, _ = load_checkpoint(cfg["train"]["checkpoint"])

probs = predict_probs(model, data.x_test[:64])
pred = probs.argmax(axis=1)
correct = np.flatnonzero(pred == data.y_test[:64])[:4]
correct

In [ ]:
methods = list(EXPLAINERS)
fig, axes = plt.subplots(len(correct), len(methods) + 1, figsize=(2.2 * (len(methods) + 1), 2.2 * len(correct)))
for row, i in enumerate(correct):
    img = data.x_test[i, 0]
    axes[row, 0].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_ylabel(f"class {pred[i]}", fontsize=8)
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    if row == 0:
        axes[row, 0].set_title("image", fontsize=9)
    for col, method in enumerate(methods, start=1):
        sal = explain(method, model, data.x_test[i], target=int(pred[i]))
        axes[row, col].imshow(img, cmap="gray", vmin=0, vmax=1)
        axes[row, col].imshow(sal, cmap="inferno", alpha=0.55)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(method, fontsize=9)
fig.tight_layout()

## Faithfulness

Pixels are deleted in the order the map ranks them and the probability of the predicted class is
tracked. A faithful map drops it fast. The dashed curve ranks pixels at random and is the control:
a map that does not beat it is not describing what the model used.

In [ ]:
from ctxaiqc.metrics import deletion_insertion

rng = np.random.default_rng(0)
i = int(correct[0])
fig, ax = plt.subplots(figsize=(5.4, 3.8))
for method in methods:
    sal = explain(method, model, data.x_test[i], target=int(pred[i]))
    out = deletion_insertion(model, data.x_test[i], sal, target=int(pred[i]), steps=32)
    ax.plot(out["fraction"], out["deletion"], label=f"{method} (AUC {out['deletion_auc']:.3f})")
random_out = deletion_insertion(
    model, data.x_test[i], rng.random(data.x_test[i, 0].shape), target=int(pred[i]), steps=32
)
ax.plot(random_out["fraction"], random_out["deletion"], "k--",
        label=f"random (AUC {random_out['deletion_auc']:.3f})")
ax.set_xlabel("fraction of pixels deleted, most salient first")
ax.set_ylabel("probability of the predicted class")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

## The negative control

The weights are re-initialised one layer at a time, from the classifier backwards, and the map is
recomputed. Agreement that stays high all the way down means the map is a property of the image and
the architecture rather than of anything the trained model learned.

This is the same logic as measuring a phantom with the source removed: if the reading does not
change, the instrument is not measuring the source.

In [ ]:
import pandas as pd

from ctxaiqc.metrics import cascading_randomization

rows = []
for method in methods:
    for record in cascading_randomization(model, data.x_test[i], EXPLAINERS[method], target=int(pred[i]), seed=0):
        rows.append({"explainer": method, **record})
sanity = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(6.4, 3.6))
for method in methods:
    sub = sanity[sanity["explainer"] == method]
    ax.plot(range(len(sub)), sub["spearman"], "o-", label=method)
ax.axhline(0, color="grey", lw=1)
ax.set_xticks(range(len(sub)))
ax.set_xticklabels(sub["layer"], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("agreement with the trained model's map")
ax.set_xlabel("layers randomised, output first")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
sanity.round(3)